# inference
Execute the cell below to run the code and save outputs directly into this notebook.

In [ ]:
import os
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt

# AAMI Int to Label Mapping
int_to_label = {
    0: 'N (Normal)', 
    1: 'S (Supraventricular)', 
    2: 'V (Ventricular)', 
    3: 'F (Fusion)', 
    4: 'Q (Unknown)'
}

def predict_heartbeat(model, segment_300_samples):
    """
    Takes a preprocessed 300-sample ECG window and uses the model 
    to decide if it implies heart disease (arrhythmia).
    """
    # 1. Ensure the shape matches exactly what the CNN-LSTM expects:
    # Model expects shape: (batch_size, time_steps, features) -> (1, 300, 1)
    if segment_300_samples.ndim == 1:
        segment = segment_300_samples.reshape(1, 300, 1)
    elif segment_300_samples.ndim == 2:
        segment = segment_300_samples.reshape(1, 300, 1)
    else:
        segment = segment_300_samples  # Already assumed to be 3D
        
    # 2. Run prediction
    prediction_probs = model.predict(segment, verbose=0)
    
    # 3. Get the most confident class
    predicted_class_int = np.argmax(prediction_probs, axis=1)[0]
    confidence = np.max(prediction_probs) * 100
    
    class_name = int_to_label[predicted_class_int]
    
    # 4. Determine if it's heart disease
    # If the class is exactly 0 ('N'), the rhythm is normal. 
    # Any other class (1, 2, 3, 4) is an Arrhythmia / abnormality!
    if predicted_class_int == 0:
        diagnosis = "No Heart Disease Detected (Normal Rhythm)"
    else:
        diagnosis = f"HEART DISEASE DETECTED (Arrhythmia Type: {class_name})"
        
    return diagnosis, confidence, predicted_class_int

def main():
    print("=== Heart Disease Inference Engine ===")
    
    model_path = r'd:\heart_disease\heart_disease_cnn_lstm.h5'
    if not os.path.exists(model_path):
        print("Model file not found! Ensure you completed Phase 4.")
        return
        
    # Load model
    print("Loading AI Model...")
    model = tf.keras.models.load_model(model_path)
    
    # To demonstrate, let's grab a random test sample we've already saved
    print("Loading a random unseen patient ECG sample...")
    X_test = np.load(r'd:\heart_disease\X_test.npy')
    y_test = np.load(r'd:\heart_disease\y_test.npy')
    
    # Pick a random sample index
    random_idx = np.random.randint(0, len(X_test))
    patient_signal = X_test[random_idx]
    true_label = y_test[random_idx]
    
    # Run Inference!
    print("\nRunning Diagnostics...")
    diagnosis, confidence, pred_class = predict_heartbeat(model, patient_signal)
    
    print("\n--- Diagnostic Report ---")
    print(f"Algorithm Diagnosis: {diagnosis}")
    print(f"AI Confidence:       {confidence:.2f}%")
    print(f"True Ground Truth:   {int_to_label[true_label]}")
    print("-------------------------\n")
    
    # Optional: Plot the signal we just analyzed
    plt.figure(figsize=(8, 4))
    plt.plot(patient_signal.flatten(), color='green' if pred_class == 0 else 'red')
    plt.title(f"ECG Heartbeat Scan \n{diagnosis}")
    plt.xlabel('Samples / Timeline')
    plt.ylabel('Amplitude (Z-score)')
    plt.grid(True)
    plt.tight_layout()
    plt.savefig('d:/heart_disease/inference_plot.png')
    print("Saved the visual scan of this heartbeat to 'd:\\heart_disease\\inference_plot.png'.")

if __name__ == '__main__':
    main()

